# 2. Basics of Optimization

## Part C Multiobjective optimization

Let's have two materials, A and B (choose as you wish from the predefined materials). 
Their total thickness is not fixed anymore.

Your task is to minimize the total thickness $l = l_A + l_B$ while maximizing the absorption.




In [ ]:
# import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from optimization_utils import *

In [ ]:
# 1. Select materials
matA = MATERIALS["acoustic_foam"]
matB = MATERIALS["melamine_foam"]

# Weight balancing thickness penalty vs absorption performance - adjust and play around to get better results
WEIGHT = 0.05 

# initial guess for the thicknesses of matA and matB
x0 = [L / 4, L / 4] 

bounds = [(0.0, L), (0.0, L)]

# Inequality constraint: L - (l_A + l_B) >= 0 (total length <= L)
constraints = [{'type': 'ineq', 'fun': lambda x: L - (x[0] + x[1])}]

def objective_scalarized(x):
    l_A, l_B = x
    if (l_A + l_B) < 1e-6:
        return 0.0
    
    _, absorption = compute_spectrum([l_A, l_B], [matA, matB])
    obj = -np.mean(absorption) + WEIGHT * ((l_A + l_B) / L)
    # obj = np.mean(np.abs(reflection_array)) + WEIGHT * ((l_A + l_B) / L)
    # obj = -np.mean(absorption_array[freqs<300]) + WEIGHT * ((l_A + l_B) / L)
    # obj = -np.max(absorption_array) + WEIGHT * ((l_A + l_B) / L)            

    return obj

res_scipy = minimize(
    objective_scalarized,
    x0,
    method='SLSQP',
    bounds=bounds,
    # constraints=constraints
)

opt_lA, opt_lB = res_scipy.x
tot_L = opt_lA + opt_lB

print(f"{matA.name} (A): {opt_lA * 1000:.2f} mm")
print(f"{matB.name} (B): {opt_lB * 1000:.2f} mm")
print(f"Total Length: {tot_L * 1000:.2f} mm")

In [ ]:
# Reflection and absorption for optimal case and single-material baselines
r_opt, abs_opt = compute_spectrum([opt_lA, opt_lB], [matA, matB])
r_matA, abs_matA = compute_spectrum([opt_lA + opt_lB], [matA])
r_matB, abs_matB = compute_spectrum([opt_lA + opt_lB], [matB])

# Plot results
plt.figure(figsize=(15, 5))
plt.subplot(121)
plt.plot(freqs, np.abs(r_opt), 'k-', linewidth=2, label=f'Part C: opt ({tot_L*1000:.1f} mm total)')
plt.plot(freqs, np.abs(r_matA), '--', label=f'Only {matA.name}')
plt.plot(freqs, np.abs(r_matB), ':', label=f'Only {matB.name}')
setup_r_axis()

plt.subplot(122)
plt.plot(freqs, abs_opt, 'k-', linewidth=2, label=f'Part C: opt ({tot_L*1000:.1f} mm total)')
plt.plot(freqs, abs_matA, '--', label=f'Only {matA.name}')
plt.plot(freqs, abs_matB, ':', label=f'Only {matB.name}')
setup_abs_axis()

plt.show()